In [1]:
%load_ext autoreload
%autoreload 2
# import torch
from pathlib import Path
import sys
import matplotlib.pyplot as plt
from matplotlib import colors
import numpy as np
import pickle
from scipy.optimize import fmin_l_bfgs_b
import json
import os
from scipy.ndimage import gaussian_filter1d
import glob
from scipy.stats import norm
# import pandas as pd
# import seaborn as sns

# # Function to find the project root directory
# for parent in [Path.cwd().resolve()] + list(Path.cwd().resolve().parents):
#     if (parent / "utils").exists():
#         sys.path.insert(0, str(parent))
#         break

# from utils.pathing import setup_repo_path
# ROOT = setup_repo_path()
# print(f"Project root set to: {ROOT}")
# -----------------------------------------------

# from utils.data import custom_optimizer

In [2]:
# --- Case study groups -------------------------------------------------------
group_2D = ['four_branch_6', 'four_branch_7', 'hat', 'himmelblau']
group_HD = ['nonlinear_oscillator', '2dof_oscillator', 'high_dimensional']

# Assign target lengths (initial 10 samples + AL iterations)
max_length = {
    case: 202 for case in group_2D
}
max_length.update({
    case: 502 for case in group_HD
})
# Combined list (preserves original order)
casestudy = group_2D + group_HD

custom_titles = [
    r'Four-branch, $k=6$', r'Four-branch, $k=7$', 'Hat', 'Himmelblau',
    r'Nonlinear oscillator', '2-DOF Oscillator', r'High-dimensional'
]

real_pf_values = {
    'four_branch_6': 0.004458488011732697,
    'four_branch_7': 0.0022232679883018138,
    'hat': 0.00038667799963150175,
    'himmelblau': 1.65E-4,
    'nonlinear_oscillator': 0.0286178,
    '2dof_oscillator': 0.0047598,
    'high_dimensional': 0.0019820
}

Define the arguments from the experiment variations

In [3]:
# Base directory of results
# base_results_dir = '/Volumes/Jonathan/MOO_results/Results/results'
base_results_dir = r'D:\\active_train\\acquisition\\MOO-AL\\notebooks\\all_results\\results_tracking'

# Active learning settings
n_exp = 15
al_strategy = [
    'moo_reliability', 'moo_knee', 'moo_compromise',
    # 'moo_eps_greedy', 
    'moo_eps_ew',
    'eff', 'u', 'erf', 'reif', 'reif2', 'portfolio'
]
al_batch = [1]
name_exp = list(range(1, n_exp + 1))

# Dictionaries to store results
output_results_dict = {}
config_results_dict = {}

# # --- Load all experiments ----------------------------------------------------
# for case in casestudy:
#     for strategy in al_strategy:
#         for batch in al_batch:
#             for exp_num in name_exp:

#                 dir_pattern = os.path.join(
#                     base_results_dir, case, f"{strategy}_{batch}_{exp_num}_*"
#                 )

#                 matching_dirs = glob.glob(dir_pattern)

#                 if not matching_dirs:
#                     print(f"No directory found for {case}, {strategy}, batch {batch}, exp {exp_num}")
#                     continue

#                 key = f"{case}_{strategy}_{batch}_{exp_num}"

#                 # Load output.json
#                 output_json_path = os.path.join(matching_dirs[0], 'output.json')
#                 if os.path.isfile(output_json_path):
#                     with open(output_json_path, 'r') as f:
#                         output_results_dict[key] = json.load(f)
#                 else:
#                     print(f"No output.json found for {key}")

#                 # Load config.json
#                 config_json_path = os.path.join(matching_dirs[0], 'config.json')
#                 if os.path.isfile(config_json_path):
#                     with open(config_json_path, 'r') as f:
#                         config_results_dict[key] = json.load(f)
#                 else:
#                     print(f"No config.json found for {key}")


Load output_dic

In [4]:
name = 'runs_14ene2026'
doe = 10  # initial DoE size (as before)
with open(f'pf_relative_error_dict_{name}.pkl', 'rb') as fp:
    loaded_file = pickle.load(fp)

relative_error_dict = loaded_file

In [5]:
incomplete_runs = []

for key, out in output_results_dict.items():
    # 1) Identify the case as the prefix of the key
    case = None
    for c in casestudy:
        if key.startswith(c + "_"):
            case = c
            break

    if case is None:
        print(f"Warning: could not identify case for key={key}")
        continue

    # 2) Remove the case + '_' prefix, then split the rest
    rest = key[len(case) + 1:]  # everything after 'case_'
    # rest should be 'strategy_batch_exp'
    try:
        strategy, batch, exp_num = rest.rsplit("_", 2)
    except ValueError:
        print(f"Warning: could not split strategy/batch/exp for key={key}")
        continue

    # 3) Now you can safely use `case` to look up max_length
    if 'Pf_model' not in out:
        incomplete_runs.append((key, "missing Pf_model"))
        continue

    pf_len = len(out['Pf_model'])
    target_len = max_length[case]-10  # 200 for 2D, 500 for HD

    if pf_len < target_len:
        incomplete_runs.append((key, pf_len))

print("\nIncomplete runs:")
for key, info in incomplete_runs:
    print(f"{key:50s} -> {info}")


Incomplete runs:


Step 1: build relative error dictionary per seed

In [ ]:
# doe = 10  # initial DoE size (as before)

# relative_error_dict = {}  # [case][strategy][exp_num] -> np.array(delta_pf)

# for case in casestudy:
#     reference_pf = float(real_pf_values[case])
#     relative_error_dict[case] = {}

#     # decide max length for this case, consistent with previous plots
#     if case in group_2D:
#         case_max_len = 200
#     else:
#         case_max_len = 500

#     for strategy in al_strategy:
#         for exp_num in range(1, n_exp + 1):
#             key = f"{case}_{strategy}_1_{exp_num}"
#             if key not in output_results_dict:
#                 continue

#             pf_model = np.asarray(
#                 output_results_dict[key].get('Pf_model', []),
#                 dtype=float
#             )

#             if pf_model.size == 0:
#                 continue

#             # truncate to desired max length (if needed)
#             max_len = min(len(pf_model), case_max_len)
#             pf_model_trunc = pf_model[:max_len]

#             # relative Pf error per iteration
#             rel_diff = np.abs(pf_model_trunc - reference_pf) / reference_pf

#             # store in nested dict
#             if strategy not in relative_error_dict[case]:
#                 relative_error_dict[case][strategy] = {}

#             relative_error_dict[case][strategy][exp_num] = rel_diff

In [ ]:
# name = 'testing'
# with open(f'relative_error_dict{name}.pkl', 'wb') as fp:
#     pickle.dump(relative_error_dict, fp)

In [ ]:
# dir = '/Users/jonathan/Documents/MOAL/Experiments/AL_StructuralReliability/'
# dir = 'D:/active_train/compromised/AL_StructuralReliability/'

# name = 'pre_results' #practical, analytical, pareto

# with open(f'relative_error_dict_{name}.pkl', 'rb') as fp:
#     loaded_file = pickle.load(fp)

# relative_error_dict = loaded_file
# doe = 10  # initial DoE size (as before)

one global minimum per strategy → top-5 strategies

In [5]:
# --- Parameters ------------------------------------------------------------
captured_ls = 5         # number of best strategies to capture (top-k)
threshold_factor = 1.0   # scale threshold up (>1) or down (<1). Set threshold_factor = 1.2 if you want the threshold to be 20% looser than that 3rd-minimum.
sigma = 1.5

# --- Step 2: per-strategy global minima, then top-k per case --------------

threshold_dict = {}  # [case] -> { "topk_minima": [...], "threshold_delta_pf": ..., "threshold_entry": ... }

for case in casestudy:
    if case not in relative_error_dict:
        continue

    # decide max length for this case
    if case in group_2D:
        case_max_len = 200
    else:
        case_max_len = 500

    per_strategy_minima = []  # one entry per strategy: its global minimum on median evolution

    # --- compute global minimum of the median evolution per strategy -------
    for strategy, exp_dict in relative_error_dict[case].items():
        if not exp_dict:
            continue

        max_len_available = max(len(arr) for arr in exp_dict.values())
        cap_len = min(max_len_available, case_max_len)

        n_runs = len(exp_dict)
        rel_diff_mat = np.full((n_runs, cap_len), np.nan, dtype=float)

        # rows = experiments
        for row_idx, (exp_num, rel_diff) in enumerate(sorted(exp_dict.items())):
            this_len = min(len(rel_diff), cap_len)
            rel_diff_mat[row_idx, :this_len] = rel_diff[:this_len]

        # median evolution across runs (ignoring NaNs)
        median_raw = np.median(rel_diff_mat, axis=0)
        # median_raw = np.mean(rel_diff_mat, axis=0)

        # median_evolution = gaussian_filter1d(median_raw, sigma=sigma)
        median_evolution = median_raw

        finite_mask = np.isfinite(median_evolution)
        if not np.any(finite_mask):
            continue

        finite_indices = np.where(finite_mask)[0]
        finite_values = median_evolution[finite_mask]

        local_argmin = np.argmin(finite_values)
        it_idx = int(finite_indices[local_argmin])

        delta_pf_min = float(median_evolution[it_idx])
        n_acquired_samples = int(doe + it_idx)

        per_strategy_minima.append({
            "strategy": strategy,
            "delta_pf_min": delta_pf_min,
            "iteration_idx": it_idx,
            "n_acquired_samples": n_acquired_samples,
        })

    if not per_strategy_minima:
        print(f"Warning: no valid minima for case '{case}'.")
        continue

    # sort strategies by their global minimum δPf
    per_strategy_minima_sorted = sorted(per_strategy_minima,
                                        key=lambda d: d["delta_pf_min"])

    # number of strategies we can actually capture
    top_k = min(captured_ls, len(per_strategy_minima_sorted))
    topk = []

    for rank in range(top_k):
        entry = per_strategy_minima_sorted[rank].copy()
        entry["rank"] = rank + 1  # 1 = best
        topk.append(entry)

    # threshold rank index (0-based): captured_ls-1, but capped by available top_k-1
    threshold_rank_index = min(captured_ls - 1, top_k - 1)
    threshold_entry = topk[threshold_rank_index]

    # apply factor to adjust strictness
    base_threshold = threshold_entry["delta_pf_min"]
    threshold_delta_pf = threshold_factor * base_threshold

    threshold_dict[case] = {
        "topk_minima": topk,                    # up to captured_ls strategies
        "captured_ls": captured_ls,
        "threshold_factor": threshold_factor,
        "threshold_delta_pf": threshold_delta_pf,
        "threshold_entry": threshold_entry      # the strategy that defines the base threshold
    }

In [7]:
from pprint import pprint
case = 'four_branch_6'
pprint(threshold_dict[case]["topk_minima"])
print("\nThreshold δPf:", threshold_dict[case]["threshold_delta_pf"])
# print("\nThreshold entry:", threshold_dict['four_branch_6']["threshold_entry"])

[{'delta_pf_min': 0.0011860738732232539,
  'iteration_idx': 109,
  'n_acquired_samples': 119,
  'rank': 1,
  'strategy': 'reif'},
 {'delta_pf_min': 0.0011914326680534687,
  'iteration_idx': 109,
  'n_acquired_samples': 119,
  'rank': 2,
  'strategy': 'portfolio'},
 {'delta_pf_min': 0.0013430528564529374,
  'iteration_idx': 176,
  'n_acquired_samples': 186,
  'rank': 3,
  'strategy': 'eff'},
 {'delta_pf_min': 0.0013932694200836418,
  'iteration_idx': 144,
  'n_acquired_samples': 154,
  'rank': 4,
  'strategy': 'erf'},
 {'delta_pf_min': 0.0014776808733412022,
  'iteration_idx': 150,
  'n_acquired_samples': 160,
  'rank': 5,
  'strategy': 'moo_knee'}]

Threshold δPf: 0.0014776808733412022


In [6]:
from typing import Optional
import numpy as np

def find_first_hit_index(rel_diff: np.ndarray,
                         threshold: float,
                         required_consecutive: int) -> Optional[int]:
    """
    Returns the first index i such that:
        rel_diff[i : i + required_consecutive] <= threshold
    for all elements in that window.
    If no such i exists, returns None.
    """
    if rel_diff.size == 0:
        return None

    is_below = rel_diff <= threshold
    max_start = rel_diff.size - required_consecutive

    if max_start < 0:
        return None

    for i in range(max_start + 1):
        # Check if all values in this window are True
        if np.all(is_below[i : i + required_consecutive]):
            return i

    return None


In [12]:
# --- Ranking all seeds per case based on threshold crossing ---------------
required_consecutive = 3

ranking_metadata = {
    "required_consecutive": required_consecutive
}
seed_ranking_dict = {}

for case in casestudy:
    if case not in relative_error_dict:
        continue
    if case not in threshold_dict:
        print(f"No threshold found for case '{case}', skipping.")
        continue

    threshold = threshold_dict[case]["threshold_delta_pf"]

    # Decide max length for this case
    if case in group_2D:
        case_max_len = 200
        no_hit_value = 201     # put non-hit seeds at the end
    else:
        case_max_len = 500
        no_hit_value = 501     # analogous idea for HD cases

    seed_entries = []

    for strategy, exp_dict in relative_error_dict[case].items():
        for exp_num, rel_diff in exp_dict.items():
            rel_diff = np.asarray(rel_diff, dtype=float)
            if rel_diff.size == 0:
                continue

            cap_len = min(rel_diff.size, case_max_len)
            rel_diff = rel_diff[:cap_len]

            # 1) best δPf
            finite_mask = np.isfinite(rel_diff)
            if not np.any(finite_mask):
                continue

            finite_values = rel_diff[finite_mask]
            best_delta_pf = float(np.min(finite_values))
            best_idx_local = int(np.where(rel_diff == best_delta_pf)[0][0])
            best_samples = int(doe + best_idx_local)

            # 2) first hit index with required_consecutive below threshold
            first_hit_idx = find_first_hit_index(
                rel_diff, threshold=threshold, required_consecutive=required_consecutive
            )

            if first_hit_idx is not None:
                reached = True
                first_hit_samples = int(doe + first_hit_idx + (required_consecutive - 1))
                delta_at_hit = float(rel_diff[first_hit_idx])
            else:
                reached = False
                first_hit_samples = no_hit_value   # 201 (2D) / 501 (HD)
                delta_at_hit = None

            seed_entries.append({
                "case": case,
                "strategy": strategy,
                "exp_num": int(exp_num),

                "reached_threshold": reached,
                "first_hit_idx": first_hit_idx,
                "first_hit_samples": first_hit_samples,
                "delta_at_hit": delta_at_hit,

                "best_delta_pf": best_delta_pf,
                "best_idx": best_idx_local,
                "best_samples": best_samples,

                "threshold_delta_pf": threshold,
            })

    if not seed_entries:
        print(f"No seed data for case '{case}'.")
        continue

    # --- Define NEW ranking rule: by first_hit_samples ---------------------
    # Priority:
    # 1) Smaller first_hit_samples (non-hit seeds already have large value)
    # 2) Among equal first_hit_samples: smaller delta_at_hit is better
    # 3) Among equal above: smaller best_delta_pf then earlier best_samples

    def seed_sort_key(entry):
        # Treat None delta_at_hit as +inf so non-hits go to the bottom on tie
        delta = entry["delta_at_hit"]
        if delta is None or not np.isfinite(delta):
            delta = np.inf
        return (
            entry["first_hit_samples"],
            delta,
            entry["best_delta_pf"],
            entry["best_samples"],
        )

    seed_entries_sorted = sorted(seed_entries, key=seed_sort_key)

    # Assign ranks
    for rank_idx, entry in enumerate(seed_entries_sorted, start=1):
        entry["global_rank"] = rank_idx

    seed_ranking_dict[case] = {
        "threshold_delta_pf": threshold,
        "required_consecutive": required_consecutive,
        "seeds": seed_entries_sorted,
    }

In [13]:
def print_seed_table_by_strategy(case):
    if case not in seed_ranking_dict:
        print(f"No ranking data for case '{case}'.")
        return

    seeds = seed_ranking_dict[case]["seeds"]

    # Group seeds by strategy
    grouped = {}
    for s in seeds:
        grouped.setdefault(s["strategy"], []).append(s)

    print(f"\n=== Seed Ranking Table for CASE: {case} ===")
    print("Threshold δPf:", seed_ranking_dict[case]["threshold_delta_pf"])
    print("Required consecutive:", seed_ranking_dict[case]["required_consecutive"])
    print("---------------------------------------------------------\n")

    # Pretty print per strategy
    for strategy in sorted(grouped.keys()):
        print(f"\n### Strategy: {strategy} ###")
        print(f"{'exp':>3} | {'rank':>4} | {'hit?':>5} | {'hit_samples':>11} | {'delta_at_hit':>13} | {'best_delta':>11} | {'best_samples':>12}")
        print("-" * 80)

        for e in grouped[strategy]:
            print(
                f"{e['exp_num']:>3} | "
                f"{e['global_rank']:>4} | "
                f"{str(e['reached_threshold']):>5} | "
                f"{str(e['first_hit_samples']):>11} | "
                f"{str(e['delta_at_hit']):>13} | "
                f"{e['best_delta_pf']:>11.3E} | "
                f"{e['best_samples']:>12}"
            )


In [14]:
print_seed_table_by_strategy('high_dimensional')


=== Seed Ranking Table for CASE: high_dimensional ===
Threshold δPf: 0.0726034308779012
Required consecutive: 3
---------------------------------------------------------


### Strategy: eff ###
exp | rank |  hit? | hit_samples |  delta_at_hit |  best_delta | best_samples
--------------------------------------------------------------------------------
 13 |    4 |  True |          82 | 0.018213925327951488 |   3.481E-03 |           82
  6 |    7 |  True |          87 | 0.05660948536831474 |   5.045E-04 |           87
  3 |   30 |  True |         116 | 0.06493440968718457 |   2.018E-03 |          122
 14 |   34 |  True |         122 | 0.0609485368314833 |   8.123E-03 |          131
  4 |   43 |  True |         127 | 0.06806256306760841 |   2.069E-03 |          136
  1 |   46 |  True |         129 | 0.053935418768920204 |   4.238E-02 |          130
 11 |   49 |  True |         133 | 0.07048435923309777 |   1.761E-02 |          132
  2 |   50 |  True |         134 | 0.01942482341069622 | 

In [206]:
# --- Compute average ranking per strategy per limit state ------------------

strategy_rankings_dict = {}  # [case] -> list of {strategy, avg_rank, ...}

for case in casestudy:
    if case not in seed_ranking_dict:
        continue
    
    seeds = seed_ranking_dict[case]["seeds"]

    # Collect ranks per strategy
    ranks_by_strategy = {}
    for s in seeds:
        strat = s["strategy"]
        ranks_by_strategy.setdefault(strat, []).append(s["global_rank"])

    # Compute average rank and number of seeds
    avg_list = []
    for strat, ranks in ranks_by_strategy.items():
        avg_rank = float(np.median(ranks))
        avg_list.append({
            "strategy": strat,
            "avg_rank": avg_rank,
            "n_seeds": len(ranks)
        })

    # Sort strategies by avg_rank (lower = better)
    avg_list_sorted = sorted(avg_list, key=lambda d: d["avg_rank"])

    strategy_rankings_dict[case] = avg_list_sorted


In [207]:
def print_strategy_ranking(case):
    if case not in strategy_rankings_dict:
        print(f"No ranking available for case '{case}'.")
        return
    
    print(f"\n=== Strategy Ranking for CASE: {case} ===")
    print("(Lower avg_rank = better performance)\n")

    for i, entry in enumerate(strategy_rankings_dict[case], start=1):
        print(f"{i:2d}. {entry['strategy']:20s} avg_rank = {entry['avg_rank']:.3f}  "
              f"(n={entry['n_seeds']})")


In [213]:
print_strategy_ranking('2dof_oscillator')


=== Strategy Ranking for CASE: 2dof_oscillator ===
(Lower avg_rank = better performance)

 1. reif2                avg_rank = 43.000  (n=15)
 2. u                    avg_rank = 46.000  (n=15)
 3. moo_eps_ew           avg_rank = 51.000  (n=15)
 4. moo_reliability      avg_rank = 63.000  (n=15)
 5. reif                 avg_rank = 66.000  (n=15)
 6. portfolio            avg_rank = 83.000  (n=15)
 7. eff                  avg_rank = 90.000  (n=15)
 8. moo_knee             avg_rank = 107.000  (n=15)
 9. erf                  avg_rank = 108.000  (n=15)
10. moo_compromise       avg_rank = 138.000  (n=15)


In [202]:
import numpy as np
import pandas as pd

all_strategies = [
    'moo_reliability', 'moo_knee', 'moo_compromise',
    'moo_eps_ew',
    'eff', 'u', 'erf', 'reif', 'reif2', 'portfolio'
]

all_limit_states = [
    'four_branch_6', 'four_branch_7',
    'hat', 'himmelblau',
    'nonlinear_oscillator', '2dof_oscillator', #'high_dimensional'
]

In [203]:
# DataFrames: rows = strategies, columns = limit states
rank_df    = pd.DataFrame(index=all_strategies, columns=all_limit_states, dtype=float)
samples_df = pd.DataFrame(index=all_strategies, columns=all_limit_states, dtype=float)

for case in all_limit_states:
    if case not in seed_ranking_dict:
        print(f"[WARN] No seed ranking data for case '{case}', skipping.")
        continue

    seeds = seed_ranking_dict[case]["seeds"]

    # Collect per-strategy data for this case
    per_strat = {}  # strategy -> { "ranks": [...], "samples": [...] }

    for s in seeds:
        strat = s["strategy"]
        if strat not in all_strategies:
            continue

        per_strat.setdefault(strat, {"ranks": [], "samples": []})
        per_strat[strat]["ranks"].append(s["global_rank"])
        # first_hit_samples already encodes non-hits as 201 / 501
        per_strat[strat]["samples"].append(s["first_hit_samples"])

    # Fill the DataFrames with means
    for strat, data in per_strat.items():
        avg_rank    = float(np.mean(data["ranks"]))
        avg_samples = float(np.mean(data["samples"]))

        rank_df.loc[strat, case]    = avg_rank
        samples_df.loc[strat, case] = avg_samples


In [204]:
# Global average rank across selected limit states (row-wise mean)
rank_df["global_avg_rank"] = rank_df.mean(axis=1, skipna=True)

# Global average first_hit_samples across selected limit states
samples_df["global_avg_samples"] = samples_df.mean(axis=1, skipna=True)

# Build final table: ranks per case + global averages + avg samples
final_df = rank_df.copy()
final_df["global_avg_samples"] = samples_df["global_avg_samples"]

# Reorder columns: global_avg_rank first, then each case, then avg samples
ordered_cols =  all_limit_states + ["global_avg_rank"]+ ["global_avg_samples"]
final_df = final_df[ordered_cols]

# Sort strategies by global_avg_rank (lower rank = better)
final_df_sorted = final_df.sort_values("global_avg_rank")


In [205]:
print("Average seed rank per strategy and limit state:\n")
print(final_df_sorted.to_string())

Average seed rank per strategy and limit state:

                 four_branch_6  four_branch_7        hat  himmelblau  nonlinear_oscillator  2dof_oscillator  global_avg_rank  global_avg_samples
portfolio            51.266667      41.000000  82.333333   72.266667             49.400000        71.133333        61.233333          164.311111
moo_eps_ew           80.133333      55.600000  67.933333   62.000000             66.466667        47.133333        63.211111          164.744444
moo_reliability      82.400000      72.866667  77.933333   57.800000             67.133333        52.533333        68.444444          173.466667
reif                 47.266667      65.933333  82.000000   83.000000             63.733333        72.600000        69.088889          187.944444
erf                  56.933333      44.933333  59.400000   97.333333             75.866667        92.066667        71.088889          205.611111
eff                  63.866667      72.400000  64.800000   83.800000             